# Receipt VLM — training on Kaggle Notebooks

Free **T4 x2 / P100**, ~30 GPU-h/week, and **up to 12 h background runs** — better
than free Colab for long jobs.

**Every epoch is checkpointed and pushed to a durable Kaggle *Dataset*.** Each snapshot is
named `phase{p}_epoch{NN}_loss{L}.pt` (phase, epoch and val-loss visible), so a session
timeout loses at most the *current* epoch — not the phase, and not the run. `/kaggle/working`
alone is wiped when an interactive session restarts; the Dataset is the durable copy.

## One-time setup (do this before running)
1. **Make the code a Dataset:** Create → *New Dataset* → upload
   `dev_ocr/vlm_training/colab_upload/receipt_vlm_colab_bundle.zip`
   (build it on your PC with `python scripts/zip_selfcontained_colab.py`).
2. **Attach it here:** right sidebar → *Add Input* → pick that dataset.
3. **Accelerator:** right sidebar → *Settings* → **GPU T4 x2** (or P100).
4. **Internet: ON** (phone-verified account — needed for pip, model downloads, and the dataset push).
5. **API token as Secrets:** Account → *Create New API Token* (downloads `kaggle.json`).
   Then *Add-ons → Secrets* → add `KAGGLE_USERNAME` and `KAGGLE_KEY` from that file.
6. For the full unattended run: **Save Version → Save & Run All (Commit)** (background up to 12 h).

> Kaggle may auto-extract the zip when you upload it as a dataset — this notebook
> handles both the zipped and the already-extracted layout.

## 0. Configuration — edit then run

In [ ]:
RUN_PHASE_1 = True
RUN_PHASE_2 = True
RUN_PHASE_3 = True
RUN_EXPORT  = True

FORCE_SMALL_BATCH = False  # set True on a single small GPU if you hit CUDA OOM

# --- durable checkpoint storage (survives session timeouts) ---
SAVE_TO_DATASET  = True               # push checkpoints to a Kaggle Dataset
SAVE_EVERY_EPOCH = True               # True: push after EVERY epoch (max safety, more uploads)
                                      # False: push once per phase (faster, loses the current phase on a crash)
DATASET_NAME     = "receipt-vlm-checkpoints"   # slug under YOUR account (lowercase + hyphens, 6-50 chars)

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Settings -> Accelerator -> GPU T4 x2"
print(torch.cuda.get_device_name(0))

## 1b. Kaggle API auth + dataset helper
Reads your `KAGGLE_USERNAME` / `KAGGLE_KEY` Secrets and defines `kaggle_save()`, which creates
the dataset the first time and adds a new version on every later call. Checkpoint files are
uploaded individually (not zipped) so a later session can resume straight from them.

In [ ]:
import json, os, subprocess
from pathlib import Path

DATASET_ID = None  # filled in by _kaggle_auth() as "<username>/<DATASET_NAME>"

def _kaggle_auth():
    global DATASET_ID
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    try:
        user = sec.get_secret("KAGGLE_USERNAME")
        key  = sec.get_secret("KAGGLE_KEY")
    except Exception as e:
        raise RuntimeError(
            "Missing Secrets. Add-ons -> Secrets -> add KAGGLE_USERNAME and KAGGLE_KEY "
            "(values from Account -> Create New API Token / kaggle.json)."
        ) from e
    kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
    (kdir / "kaggle.json").write_text(json.dumps({"username": user, "key": key}))
    os.chmod(kdir / "kaggle.json", 0o600)
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
    DATASET_ID = f"{user}/{DATASET_NAME}"

def kaggle_save(folder, msg):
    """Create-or-version a Kaggle Dataset from `folder`. No-op if SAVE_TO_DATASET is False."""
    if not SAVE_TO_DATASET:
        return
    folder = Path(folder)
    files = [p for p in folder.glob("*.pt") if p.is_file()]
    if not files:
        return
    (folder / "dataset-metadata.json").write_text(json.dumps({
        "title": DATASET_NAME, "id": DATASET_ID, "licenses": [{"name": "CC0-1.0"}],
    }))
    exists = subprocess.run(["kaggle", "datasets", "files", DATASET_ID],
                            capture_output=True, text=True).returncode == 0
    sub = "version" if exists else "create"
    cmd = ["kaggle", "datasets", sub, "-p", str(folder), "--dir-mode", "skip"]
    if exists:
        cmd += ["-m", msg]
    print(f"   [dataset] {sub}: {msg}", flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("   [dataset] WARNING push failed:", r.stderr.strip()[:300], flush=True)
    else:
        print(f"   [dataset] -> https://www.kaggle.com/datasets/{DATASET_ID}", flush=True)

if SAVE_TO_DATASET:
    _kaggle_auth()
    print("Checkpoints -> dataset:", DATASET_ID, "| every epoch:" , SAVE_EVERY_EPOCH)
else:
    print("SAVE_TO_DATASET=False -- /kaggle/working only (lost on interactive timeout unless you Commit)")

## 2. Get the code from your attached dataset
Copies the bundle into the writable `/kaggle/working` (editable installs and
`*.egg-info` cannot be written under the read-only `/kaggle/input`).

In [ ]:
import glob, os, shutil, zipfile
from pathlib import Path

WORK = Path("/kaggle/working/repo")

def materialize():
    # A) zip uploaded as a dataset (not auto-extracted)
    zips = glob.glob("/kaggle/input/**/receipt_vlm_colab_bundle.zip", recursive=True)
    if zips:
        WORK.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(WORK)
        return
    # B) dataset already extracted -> copy the dev_ocr tree into WORK
    hits = glob.glob("/kaggle/input/**/vlm_training/scripts/train.py", recursive=True)
    if hits:
        dev_ocr_src = Path(hits[0]).resolve().parents[2]
        dest = WORK / "dev_ocr"
        if not dest.exists():
            shutil.copytree(dev_ocr_src, dest)
        return
    raise FileNotFoundError("Bundle not found in /kaggle/input -- did you Add Input?")

if not list(WORK.glob("**/vlm_training/scripts/train.py")):
    materialize()

hits = glob.glob(str(WORK / "**/vlm_training/scripts/train.py"), recursive=True)
assert hits, "train.py not found after materialize"
TRAIN_PKG = Path(hits[0]).resolve().parents[1]
DEV_OCR = TRAIN_PKG.parent
os.chdir(TRAIN_PKG)
print("Train package:", TRAIN_PKG)

## 3. Install dependencies (~2-3 min)

In [ ]:
import subprocess, sys
def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

pip("-r", "requirements-training.txt", "tokenizers>=0.22,<=0.23")
pip("-e", str(DEV_OCR))        # receipt_ocr
pip("-e", str(TRAIN_PKG))      # receipt_vlm
print("Install OK")

## 4. Point configs at the Kaggle working dir
Training writes to `/kaggle/working/checkpoints` (fast local disk); cell 5 mirrors new
checkpoints to your Kaggle Dataset as they appear.

In [ ]:
import yaml
from pathlib import Path

CKPT_DIR = Path("/kaggle/working/checkpoints"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
cfg_path = TRAIN_PKG / "configs" / "colab_paths.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) or {}
cfg["checkpoint_dir"] = str(CKPT_DIR)
cfg["log_every"] = 25
if FORCE_SMALL_BATCH:
    cfg["batch_size"] = 4
cfg.setdefault("data", {})
cfg["data"]["real_images_dir"] = str(DEV_OCR / "data" / "raw" / "images_tickets_caisse")
cfg["data"]["real_labels_dir"] = str(TRAIN_PKG / "data" / "real_labels")
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=False))
print(cfg_path.read_text())

## 4b. Resume across sessions

To continue an interrupted run: **Add Input** → attach your `receipt-vlm-checkpoints` dataset,
then run this cell. It copies **every** `phase*.pt` snapshot back into the working dir, and
`train.py` then **auto-resumes**:

- a phase that stopped mid-way restarts from its **latest epoch** snapshot (no lost epochs);
- a finished phase seeds the next one from its **last** checkpoint.

You usually don't even need to flip `RUN_PHASE_*` — finished phases fast-forward and skip
themselves. (Set them `False` only to force-skip.) Checkpoints are ~45 MB each.

In [ ]:
import glob, os, shutil
from pathlib import Path

restored = []
for src in glob.glob("/kaggle/input/**/phase*.pt", recursive=True):
    dest = CKPT_DIR / os.path.basename(src)
    if not dest.exists():
        shutil.copy(src, dest)
        restored.append(dest.name)
print(f"Restored {len(restored)} checkpoint(s).")
for name in sorted(restored):
    print("  ", name)
if not restored:
    print("  (nothing in /kaggle/input -- fresh run, or you forgot to Add Input)")

## 5. Train phases 1 -> 2 -> 3
Live timestamp/gap heartbeat. `train.py` auto-resumes from whatever is already in the
checkpoint dir (see cell 4b). With `SAVE_EVERY_EPOCH=True`, each new
`phase{p}_epoch{NN}_loss{L}.pt` is pushed to the Dataset the moment it's written — so a crash
costs at most the current epoch. ~3-4 h on T4 (silent startup downloads CLIP+SmolLM2 then CORD).

> Per-epoch pushes each create a new dataset version and re-upload the (growing) checkpoint
> folder. If that overhead bites, set `SAVE_EVERY_EPOCH=False` in cell 0 to push once per phase.

In [ ]:
import subprocess, sys, os, re, time, datetime

p3 = f"{CKPT_DIR}/phase3_best.pt"   # used by the export cell

def _have_phase(p):
    d = Path(CKPT_DIR)
    return bool(list(d.glob(f"phase{p}_best.pt")) or list(d.glob(f"phase{p}_epoch*.pt")))

def run_train(config):
    cmd = [sys.executable, "-u", "scripts/train.py", "--config", config]
    print("\n>>", " ".join(cmd), flush=True)
    print("   (startup is silent for a few min: downloading CLIP+SmolLM2, then CORD)", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    start = last = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    for line in proc.stdout:
        now = time.time()
        gap = now - last; last = now
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[{ts} +{int(now-start):>5}s gap{gap:4.0f}s] {line}", end="", flush=True)
        # Mirror each per-epoch snapshot to the Dataset as soon as the trainer prints it.
        if SAVE_TO_DATASET and SAVE_EVERY_EPOCH and "Checkpoint saved" in line and "_epoch" in line:
            m = re.search(r"(phase\d+_epoch\d+_loss[\d.]+\.pt)", line)
            kaggle_save(CKPT_DIR, m.group(1) if m else os.path.basename(config))
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{config} failed (exit {proc.returncode})")
    print(f"-- {config} done in {int(time.time()-start)}s", flush=True)
    if SAVE_TO_DATASET and not SAVE_EVERY_EPOCH:   # per-epoch path already pushed
        kaggle_save(CKPT_DIR, f"after {os.path.basename(config)}")

# Force-skipping a phase requires a checkpoint for it (run cell 4b first).
if not RUN_PHASE_1 and (RUN_PHASE_2 or RUN_PHASE_3):
    assert _have_phase(1), "no phase-1 checkpoint -- run cell 4b or set RUN_PHASE_1=True"
if not RUN_PHASE_2 and RUN_PHASE_3:
    assert _have_phase(2), "no phase-2 checkpoint -- run cell 4b or set RUN_PHASE_2=True"

if RUN_PHASE_1:
    run_train("configs/phase1_colab.yaml")
if RUN_PHASE_2:
    run_train("configs/phase2_colab.yaml")
if RUN_PHASE_3:
    run_train("configs/phase3_colab.yaml")
print("Training done")

## 6. Export merged inference checkpoint

In [ ]:
MERGED = f"{CKPT_DIR}/receipt_vlm_500m_merged.pt"
if RUN_EXPORT:
    subprocess.check_call([sys.executable, "scripts/export_checkpoint.py",
                           "--checkpoint", p3, "--output", MERGED])
    print("Merged ->", MERGED)
    kaggle_save(CKPT_DIR, "merged inference checkpoint")
else:
    print("Export skipped")gity

## 7. Get your files
All snapshots are durable in your **`receipt-vlm-checkpoints` Dataset**
(`https://www.kaggle.com/datasets/<you>/receipt-vlm-checkpoints` → *Download*), and also under
`/kaggle/working` if you ran via *Save & Run All*. Grab `receipt_vlm_500m_merged.pt` for
local inference; the `phase*_epoch*.pt` snapshots let you resume or pick a specific epoch.

In [ ]:
from pathlib import Path
print("Files in /kaggle/working/checkpoints:")
for p in sorted(Path(CKPT_DIR).glob("*.pt")):
    print(f"  {p.name:40} {p.stat().st_size/1e6:8.1f} MB")
if SAVE_TO_DATASET and DATASET_ID:
    print("\nDurable copy -> https://www.kaggle.com/datasets/" + DATASET_ID)

## 8. (optional) Sanity check on one photo

In [ ]:
import json, os
from pathlib import Path

photos = sorted((DEV_OCR / "data" / "raw" / "images_tickets_caisse").glob("*.jpg"))
if photos and Path(MERGED).is_file():
    os.environ.update({
        "RECEIPT_OCR_BACKEND": "vlm",
        "RECEIPT_VLM_MODEL": "receipt-vlm-500m",
        "RECEIPT_VLM_MODE": "json",
        "RECEIPT_VLM_MODEL_PATH": MERGED,
    })
    from receipt_ocr import extract_receipt
    print(json.dumps(extract_receipt(str(photos[0])), indent=2, ensure_ascii=False)[:1200])
else:
    print("Need merged checkpoint + at least one photo")